# Robustesse des seuils + IC bootstrap (reproductible)

Analyse de robustesse de la taxonomie A1-A7 par **une seule fonction** librairie, `gbfs_toolkit.audit_sensitivity`, et IC par `gbfs_toolkit.flag_rate_ci` (seed 42). Désamorce la critique « seuils inductifs calibrés sur les données » : on montre que l'ensemble flaggé est invariant ou sur un plateau de stabilité sur les grilles de seuils, et que chaque taux porte un IC.

France : depuis le parquet certifié (hors-ligne, exact). Monde : depuis l'archive gelée `experiments/unified_audit/raw_world` (variante operator).

In [1]:
import sys; sys.path.insert(0, '.')
import glob, pandas as pd
import gbfs_toolkit as gb
import unified_audit as ua
GRIDS = {'a4_sigma':[2.0,2.5,3.0,3.5,4.0], 'a5_area_km2':[30000,40000,50000,60000,70000],
         'a6_tau':[0.005,0.01,0.02,0.05], 'a7_tau':[0.3,0.4,0.5,0.6,0.7], 'n_min':[10,15,20,25,30]}
print('toolkit', gb.__version__)

toolkit 1.4.0


## 1. France — robustesse (Jaccard min vs baseline)

In [2]:
fr = pd.read_parquet('../catalogue/stations_gold_standard_final.parquet')
fr_raw = fr[['system_id','station_id','station_type','capacity','lat','lon']]
sfr = gb.audit_sensitivity(fr_raw, GRIDS, a7_scope='all')
fr_pivot = sfr.pivot_table(index='param', columns='class', values='jaccard_vs_baseline', aggfunc='min').round(3)
fr_pivot[['A1','A2','A3','A4','A5','A6','A7']]

class,A1,A2,A3,A4,A5,A6,A7
param,,,,,,,
a4_sigma,1.0,1.0,1.0,0.909,1.000,1.0,1.00
a5_area_km2,1.0,1.0,1.0,1.000,0.667,1.0,1.00
a6_tau,1.0,1.0,1.0,1.000,1.000,1.0,1.00
a7_tau,1.0,1.0,1.0,1.000,1.000,1.0,0.97
n_min,1.0,1.0,1.0,1.000,1.000,1.0,1.00


## 2. Monde (n=936) — robustesse

In [3]:
frames = [pd.read_parquet(f) for f in glob.glob('../experiments/unified_audit/raw_world/*.parquet')]
world = pd.concat(frames, ignore_index=True)
cat = ua.load_catalog('../experiments/e2_threshold_sensitivity/mobilitydata_systems.csv')
name_of = dict(zip(cat.system_id.astype(str), cat.get('name')))
world = ua.apply_operator_types(world, name_of)
sw = gb.audit_sensitivity(world, GRIDS, a7_scope='all')
w_pivot = sw.pivot_table(index='param', columns='class', values='jaccard_vs_baseline', aggfunc='min').round(3)
w_pivot[['A1','A2','A3','A4','A5','A6','A7']]

class,A1,A2,A3,A4,A5,A6,A7
param,,,,,,,
a4_sigma,1.0,1.000,1.0,0.947,1.0,1.0,1.000
a5_area_km2,1.0,1.000,1.0,1.000,0.8,1.0,1.000
a6_tau,1.0,1.000,1.0,1.000,1.0,1.0,1.000
a7_tau,1.0,1.000,1.0,1.000,1.0,1.0,0.960
n_min,1.0,0.375,1.0,1.000,1.0,1.0,0.855


Lecture : A1-A3/A6 invariants (Jaccard 1,0) ; A4/A7 sur un plateau (≥0,91 FR, ≥0,95 monde sur σ ; ≥0,96 sur τ_A7). `n_min` a plus de levier mondialement (A2/A7), nuance honnête à reporter.

## 3. Détail A4(σ) et A7(τ) — comptes par valeur (monde)

In [4]:
for p in ['a4_sigma','a7_tau']:
    t = sw[sw.param==p].pivot(index='value', columns='class', values='systems_flagged')
    cols = ['A4','A5','A7'] if p=='a4_sigma' else ['A7']
    print(f'-- {p} --'); print(t[cols]); print()

-- a4_sigma --
class   A4  A5   A7
value              
2.0    544  30  288
2.5    525  30  288
3.0    515  30  288
3.5    504  30  288
4.0    491  30  288

-- a7_tau --
class   A7
value     
0.3    300
0.4    295
0.5    288
0.6    286
0.7    281



## 4. IC95 bootstrap par cluster (seed 42)

In [5]:
ci_fr = gb.flag_rate_ci(gb.audit_static(fr_raw, a7_scope='all'), seed=42)
ci_w  = gb.flag_rate_ci(gb.audit_static(world, a7_scope='all'), seed=42)
comp = ci_fr.merge(ci_w, on='class', suffixes=('_FR','_World'))
comp[['class','rate_FR','ci_lo_FR','ci_hi_FR','rate_World','ci_lo_World','ci_hi_World']].round(3)

,class,rate_FR,ci_lo_FR,ci_hi_FR,rate_World,ci_lo_World,ci_hi_World
0,A1,0.138,0.081,0.203,0.038,0.027,0.051
1,A2,0.008,0.000,0.024,0.008,0.003,0.015
2,A3,0.333,0.252,0.415,0.747,0.719,0.775
3,A4,0.715,0.634,0.797,0.550,0.518,0.582
4,A5,0.032,0.008,0.065,0.032,0.021,0.044
5,A6,0.000,0.000,0.000,0.000,0.000,0.000
6,A7,0.260,0.187,0.342,0.308,0.279,0.339


## 5. Figure — stabilité A4 vs σ (FR et monde)

In [6]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fa = sfr[(sfr.param=='a4_sigma')&(sfr['class']=='A4')]
wa = sw[(sw.param=='a4_sigma')&(sw['class']=='A4')]
fig, ax = plt.subplots(figsize=(6,3))
ax.plot(fa.value, fa.jaccard_vs_baseline, 'o-', label='France')
ax.plot(wa.value, wa.jaccard_vs_baseline, 's-', label='Monde (n=936)')
ax.axvline(3.0, ls='--', color='grey', lw=1)
ax.set_xlabel('a4_sigma'); ax.set_ylabel('Jaccard A4 vs baseline'); ax.set_ylim(0.8,1.02)
ax.set_title('Plateau de stabilité A4 autour de σ=3'); ax.legend()
plt.tight_layout(); plt.show()

/tmp/ipykernel_37537/31067860.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Reproductibilité

```
gb.audit_sensitivity(stations, GRIDS, a7_scope='all')   # déterministe
gb.flag_rate_ci(verdict, seed=42)                       # IC reproductible
```
Aucun script externe : robustesse et IC sortent de la librairie publiée, donc citables et rejouables.